# 06 · Números Canônicos — Projeto Vértice (Vértice Retail)

**Papel deste notebook:** ser a **fonte única de verdade** dos números do diagnóstico.
Cada valor citado em qualquer outro artefato — o notebook de impacto (`07`), o sanity
check (`05`) e a dashboard HTML — deve vir **daqui**, de uma célula com fórmula e coluna
de origem explícitas. Se um número não está neste notebook, ele não é canônico.

**Por que isto existe:** antes deste notebook, os valores viviam espalhados (notebooks
03/04, `metodologia_impacto.md`) sob janelas diferentes, e ninguém tinha reconciliado.
O resultado eram dois números para a mesma coisa. Aqui a janela é fixa, o cálculo é um só,
e tudo é exportado para `outputs/numeros_canonicos.json`.

**Convenção de janela (decidida no notebook 01, §6):**
- **Vendas → 13 meses** (01/01/2023 a 26/01/2024, base cheia de 27.758 pedidos válidos).
- **Atendimento → 36 meses** (2023–2025 completos).
- **Anualização:** impactos de vendas viram taxa anual por `× 365/dias_da_janela`;
  impactos de atendimento por `÷ 3`. **Não há projeção "base completa" (×2,88)** — só
  escala observada e anualizada.

**Relação com os outros notebooks:** este notebook **reproduz** a lógica de limpeza do
`01` e os testes do `03` — não redefine nada. É a consolidação, não uma nova análise.


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy import stats

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

RAW_DIR = Path('..') / 'Dados do Case'
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)

def carregar(nome):
    # encoding utf-8-sig remove o BOM presente em 4 das 5 bases (ver nb01, seção 2)
    return pd.read_csv(RAW_DIR / f'[BootCamp EloGroup 2026] {nome}.csv', encoding='utf-8-sig')

vendas_raw  = carregar('Vendas')
atendimento = carregar('Atendimento').dropna(subset=['customer_id']).copy()
estoque     = carregar('Estoque')
clientes    = carregar('Clientes')
marketing   = carregar('Marketing')

# --- limpeza de vendas: mesma decisão do nb01 (descarta 1 linha vazia, deriva colunas) ---
v = vendas_raw.dropna(subset=['quantidade']).copy()
v['data_pedido'] = pd.to_datetime(v['data_pedido'])
v['devolvido']   = v['devolvido'].astype(bool)
v['margem_pct']  = v['margem_contribuicao'] / v['receita_liquida']
v['desconto_pct']= np.where(v['receita_bruta'] > 0, v['desconto_reais'] / v['receita_bruta'], np.nan)
atendimento['data_abertura'] = pd.to_datetime(atendimento['data_abertura'])

DIAS = (v['data_pedido'].max() - v['data_pedido'].min()).days   # janela de vendas
ANU  = 365.0 / DIAS                                             # fator de anualização (vendas)
print(f'Janela de vendas: {DIAS} dias  |  fator de anualização ANU = {ANU:.5f}')
print(f'Pedidos válidos: {len(v)}  |  Tickets de atendimento: {len(atendimento)}')

# --- registro canônico: cada número guarda valor, unidade, janela, fonte e fórmula ---
CANON = {}
def reg(chave, valor, unidade, janela, fonte, formula):
    CANON[chave] = {'valor': valor, 'unidade': unidade, 'janela': janela,
                    'fonte': fonte, 'formula': formula}
    print(f'  {chave:38} = {valor} {unidade}')
    return valor

Janela de vendas: 390 dias  |  fator de anualização ANU = 0.93590
Pedidos válidos: 27758  |  Tickets de atendimento: 35840


## 1. Base e janela

Totais financeiros da janela de 13 meses e a margem **contábil** (a que a coluna
`margem_contribuicao` reporta, sobre toda a receita líquida).

In [2]:
rb = v['receita_bruta'].sum(); rl = v['receita_liquida'].sum(); mc = v['margem_contribuicao'].sum()
reg('pedidos_validos', int(len(v)), 'pedidos', '13m', 'Vendas.csv', 'len após dropna(quantidade)')
reg('janela_dias', int(DIAS), 'dias', '13m', 'Vendas.data_pedido', '(max-min).days')
reg('pedidos_2023', int((v['data_pedido'].dt.year==2023).sum()), 'pedidos', '2023', 'Vendas.data_pedido', 'ano==2023')
reg('receita_bruta_total', round(rb,2), 'R$', '13m', 'Vendas.receita_bruta', 'soma')
reg('receita_liquida_total', round(rl,2), 'R$', '13m', 'Vendas.receita_liquida', 'soma')
reg('margem_contribuicao_total', round(mc,2), 'R$', '13m', 'Vendas.margem_contribuicao', 'soma')
reg('margem_contabil_pct', round(100*mc/rl,2), '%', '13m', 'Vendas', 'margem_contribuicao / receita_liquida')
None

  pedidos_validos                        = 27758 pedidos
  janela_dias                            = 390 dias
  pedidos_2023                           = 26538 pedidos
  receita_bruta_total                    = 20526133.84 R$
  receita_liquida_total                  = 18889334.01 R$
  margem_contribuicao_total              = 10270436.65 R$
  margem_contabil_pct                    = 54.37 %


## 2. Margem realizada — o maior número do diagnóstico

A `margem_contribuicao` é uma identidade contábil: conta pedidos que **nunca viraram
caixa** (devolvidos, cancelados, aguardando pagamento). A margem **realizada** recalcula
o numerador só com pedidos aprovados e não devolvidos, sobre a **mesma** receita líquida.
Mesma lógica do Teste 15 do notebook 03.

In [3]:
realizado = v[(~v['devolvido']) & (v['status_pagamento']=='Aprovado')]
mr  = realizado['margem_contribuicao'].sum()
gap = mc - mr
reg('margem_realizada_rs', round(mr,2), 'R$', '13m', 'Vendas (aprovado & não devolvido)', 'soma margem dos pedidos que viraram caixa')
reg('margem_realizada_pct', round(100*mr/rl,2), '%', '13m', 'Vendas', 'margem_realizada / receita_liquida_total')
reg('margem_realizada_ano', round(mr*ANU,2), 'R$/ano', 'anualizado', 'Vendas', 'margem_realizada_13m * 365/dias — DENOMINADOR DA RÉGUA')
reg('margem_gap_rs_13m', round(gap,2), 'R$', '13m', 'Vendas', 'margem_contabil - margem_realizada')
reg('margem_gap_rs_ano', round(gap*ANU,2), 'R$/ano', 'anualizado', 'Vendas', 'gap_13m * 365/dias')
reg('pct_pedidos_nao_caixa', round(100*(1-len(realizado)/len(v)),2), '%', '13m', 'Vendas', '1 - realizados/total')
reg('frete_afundado_devolvidos_rs_13m', round(v.loc[v['devolvido'],'custo_frete'].sum(),2), 'R$', '13m', 'Vendas.custo_frete (devolvido)', 'frete de saída já pago em devolvidos (custo afundado real; a base não tem frete reverso)')
None

  margem_realizada_rs                    = 7706720.46 R$
  margem_realizada_pct                   = 40.8 %
  margem_realizada_ano                   = 7212699.92 R$/ano
  margem_gap_rs_13m                      = 2563716.19 R$
  margem_gap_rs_ano                      = 2399375.41 R$/ano
  pct_pedidos_nao_caixa                  = 25.01 %
  frete_afundado_devolvidos_rs_13m       = 51006.15 R$


## 3. Desconto

Desconto total concedido na janela. O notebook 04 mostra que **unidades por pedido não
sobem com o desconto** — por isso cada real de desconto é tratado como margem cedida 1:1
no notebook de impacto (é um **teto**, não uma projeção de recuperação).

Faixas de desconto usadas (as mesmas da dashboard): `sem desconto` isolada, depois
5-10 / 10-15 / 15-20 / 20-25 / 25%+.

In [4]:
desc = v['desconto_reais'].sum()
reg('desconto_total_rs_13m', round(desc,2), 'R$', '13m', 'Vendas.desconto_reais', 'soma')
reg('desconto_total_rs_ano', round(desc*ANU,2), 'R$/ano', 'anualizado', 'Vendas.desconto_reais', 'soma_13m * 365/dias')
reg('desconto_pct_receita_bruta', round(100*desc/rb,2), '%', '13m', 'Vendas', 'desconto / receita_bruta')
reg('desconto_pct_pedidos', round(100*(v['desconto_reais']>0).mean(),2), '%', '13m', 'Vendas', 'share de pedidos com desconto > 0')

faixas_bins = [-0.001, 0.0001, 0.10, 0.15, 0.20, 0.25, 1.0]
faixas_lab  = ['sem desconto','5-10%','10-15%','15-20%','20-25%','25%+']
v['faixa_desconto'] = pd.cut(v['desconto_pct'], bins=faixas_bins, labels=faixas_lab)
por_faixa = v.groupby('faixa_desconto', observed=True).agg(
    margem_pct=('margem_pct','mean'), unidades=('quantidade','mean'), pedidos=('order_id','size'))
print(por_faixa.round(4))
reg('unidades_amplitude_faixas', round(por_faixa['unidades'].max()-por_faixa['unidades'].min(),3), 'unidades', '13m', 'Vendas.quantidade por faixa', 'max-min da média de unidades entre faixas (≈0 → desconto não compra volume)')
reg('margem_faixa_sem_desconto_pct', round(100*por_faixa.loc['sem desconto','margem_pct'],1), '%', '13m', 'Vendas', 'margem média da faixa sem desconto')
reg('margem_faixa_top_pct', round(100*por_faixa.loc['25%+','margem_pct'],1), '%', '13m', 'Vendas', 'margem média da faixa 25%+')
reg('desconto_spread_pp', round(100*(por_faixa.loc['sem desconto','margem_pct']-por_faixa.loc['25%+','margem_pct']),1), 'p.p.', '13m', 'Vendas', 'margem(sem desconto) - margem(25%+)')
reg('pedidos_faixa_top', int(por_faixa.loc['25%+','pedidos']), 'pedidos', '13m', 'Vendas', 'nº de pedidos na faixa 25%+')
None

  desconto_total_rs_13m                  = 1636799.83 R$
  desconto_total_rs_ano                  = 1531876.76 R$/ano
  desconto_pct_receita_bruta             = 7.97 %
  desconto_pct_pedidos                   = 35.49 %
                margem_pct  unidades  pedidos
faixa_desconto                               
sem desconto        0.5512    3.5113    17908
5-10%               0.5154    3.5247     1376
10-15%              0.4721    3.5047     1373
15-20%              0.4452    3.5535     1355
20-25%              0.4095    3.4099     1459
25%+                0.3126    3.5195     4287
  unidades_amplitude_faixas              = 0.144 unidades
  margem_faixa_sem_desconto_pct          = 55.1 %
  margem_faixa_top_pct                   = 31.3 %
  desconto_spread_pp                     = 23.9 p.p.
  pedidos_faixa_top                      = 4287 pedidos


## 4. Marketplace — margem e frete

**Nota de método (importante):** há duas leituras legítimas e diferentes destas métricas.
Para o **R$ de impacto** usamos a versão **agregada** (revenue-weighted), que é a correta
para multiplicar por receita. A **mediana por pedido** (margem) e a **média das razões por
pedido** (frete) são estatísticas descritivas usadas nos testes do nb03/nb04 — guardadas
aqui com rótulo próprio para nunca serem confundidas com as agregadas.

In [5]:
mkt = v[v['canal']=='Marketplace']; out = v[v['canal']!='Marketplace']
m_mkt = 100*mkt['margem_contribuicao'].sum()/mkt['receita_liquida'].sum()
m_out = 100*out['margem_contribuicao'].sum()/out['receita_liquida'].sum()
gap_pp = m_out - m_mkt
gap_rs = gap_pp/100 * mkt['receita_liquida'].sum()
reg('mkt_margem_agregada_pct', round(m_mkt,2), '%', '13m', 'Vendas Marketplace', 'sum margem / sum receita_liq (AGREGADO — usar para R$)')
reg('mkt_margem_demais_pct', round(m_out,2), '%', '13m', 'Vendas demais canais', 'sum margem / sum receita_liq (agregado)')
reg('mkt_margem_mediana_por_pedido_pct', round(100*mkt['margem_pct'].median(),2), '%', '13m', 'Vendas Marketplace', 'mediana de margem_pct por pedido (estatística de teste, nb03)')
reg('mkt_gap_pp', round(gap_pp,2), 'p.p.', '13m', 'Vendas', 'margem_demais - margem_mkt (agregado)')
reg('mkt_gap_rs_13m', round(gap_rs,2), 'R$', '13m', 'Vendas', 'gap_pp * receita_liq_marketplace')
reg('mkt_gap_rs_ano', round(gap_rs*ANU,2), 'R$/ano', 'anualizado', 'Vendas', 'gap_13m * 365/dias')
reg('mkt_frete_pct_agregado', round(100*mkt['custo_frete'].sum()/mkt['receita_liquida'].sum(),2), '%', '13m', 'Vendas Marketplace', 'sum frete / sum receita_liq (AGREGADO)')
reg('mkt_frete_pct_media_por_pedido', round(100*(mkt['custo_frete']/mkt['receita_liquida']).mean(),2), '%', '13m', 'Vendas Marketplace', 'média da razão frete/receita por pedido (estatística de teste, nb04)')
reg('mkt_frete_medio_rs', round(mkt['custo_frete'].mean(),2), 'R$', '13m', 'Vendas Marketplace', 'média custo_frete por pedido')
reg('mkt_pct_pedidos_com_frete', round(100*(mkt['custo_frete']>0).mean(),0), '%', '13m', 'Vendas Marketplace', 'share de pedidos com frete > 0')
None

  mkt_margem_agregada_pct                = 51.42 %
  mkt_margem_demais_pct                  = 55.16 %
  mkt_margem_mediana_por_pedido_pct      = 49.64 %
  mkt_gap_pp                             = 3.74 p.p.
  mkt_gap_rs_13m                         = 148967.1 R$
  mkt_gap_rs_ano                         = 139417.93 R$/ano
  mkt_frete_pct_agregado                 = 4.93 %
  mkt_frete_pct_media_por_pedido         = 9.67 %
  mkt_frete_medio_rs                     = 32.57 R$
  mkt_pct_pedidos_com_frete              = 100.0 %


## 5. Atendimento (36 meses)

Custo de atendimento e economia do ChatBot. Base de 36 meses → todos os valores anuais
são `÷ 3`. Mesma lógica do Teste 16/17 do notebook 03.

In [6]:
falha = ['Onde está meu pedido?', 'Defeito', 'Pagamento não aprovado']
custo_total = atendimento['custo_operacional_ticket'].sum()
custo_falha = atendimento.loc[atendimento['categoria_problema'].isin(falha),'custo_operacional_ticket'].sum()
reg('atend_custo_total_ano', round(custo_total/3,2), 'R$/ano', '36m÷3', 'Atendimento.custo_operacional_ticket', 'soma_36m / 3')
reg('atend_custo_falha_ano', round(custo_falha/3,2), 'R$/ano', '36m÷3', 'Atendimento (falha operacional)', 'soma_falha_36m / 3')
reg('atend_pct_falha_volume', round(100*atendimento['categoria_problema'].isin(falha).mean(),2), '%', '36m', 'Atendimento.categoria_problema', 'share de tickets de falha operacional')

sub = atendimento[(atendimento['categoria_problema']=='Onde está meu pedido?') & (atendimento['canal_entrada']!='ChatBot')]
custo_atual = sub['custo_operacional_ticket'].sum(); custo_bot = len(sub)*2.0; eco = custo_atual - custo_bot
reg('chatbot_tickets_migraveis', int(len(sub)), 'tickets', '36m', "Atendimento ('onde está meu pedido' & != ChatBot)", 'contagem')
reg('chatbot_economia_36m', round(eco,2), 'R$', '36m', 'Atendimento', 'custo_atual - nº*R$2,00')
reg('chatbot_economia_ano', round(eco/3,2), 'R$/ano', '36m÷3', 'Atendimento', 'economia_36m / 3')
reg('chatbot_csat', round(atendimento.loc[atendimento['canal_entrada']=='ChatBot','nota_csat'].mean(),3), 'nota', '36m', 'Atendimento.nota_csat ChatBot', 'média')
reg('humano_csat', round(atendimento.loc[atendimento['canal_entrada']!='ChatBot','nota_csat'].mean(),3), 'nota', '36m', 'Atendimento.nota_csat humano', 'média')
reg('chatbot_custo_ticket', round(atendimento.loc[atendimento['canal_entrada']=='ChatBot','custo_operacional_ticket'].mean(),2), 'R$', '36m', 'Atendimento ChatBot', 'média custo/ticket')
reg('humano_custo_ticket', round(atendimento.loc[atendimento['canal_entrada']!='ChatBot','custo_operacional_ticket'].mean(),2), 'R$', '36m', 'Atendimento humano', 'média custo/ticket')
None

  atend_custo_total_ano                  = 177420.0 R$/ano
  atend_custo_falha_ano                  = 106702.67 R$/ano
  atend_pct_falha_volume                 = 60.14 %
  chatbot_tickets_migraveis              = 8560 tickets
  chatbot_economia_36m                   = 138130.0 R$
  chatbot_economia_ano                   = 46043.33 R$/ano
  chatbot_csat                           = 3.265 nota
  humano_csat                            = 3.235 nota
  chatbot_custo_ticket                   = 2.0 R$
  humano_custo_ticket                    = 18.06 R$


## 6. Operações — ruptura e estoque parado

Ruptura em SKUs de alto giro (curva A por Pareto de receita) é **receita em risco**, não
perda medida — a base de estoque é uma foto, sem histórico de dias sem estoque. O capital
imobilizado é calculado e **descartado** (magnitude incompatível com a receita → artefato).

In [7]:
sku_receita = v.groupby('sku_id')['receita_liquida'].sum().sort_values(ascending=False)
curva = pd.cut(sku_receita.cumsum()/sku_receita.sum(), bins=[-0.01,0.80,0.95,1.0], labels=['A','B','C'])
skus_a = curva[curva=='A'].index
crit_a = estoque[estoque['sku_id'].isin(skus_a) & estoque['status_disponibilidade'].isin(['Ruptura','Estoque Crítico'])]
rec_crit = sku_receita.loc[sku_receita.index.isin(crit_a['sku_id'])].sum()
reg('ruptura_curvaA_qtd', int(len(crit_a)), 'SKUs', '13m', 'Estoque × Vendas', 'curva A em Ruptura/Crítico')
reg('ruptura_curvaA_receita_dia', round(rec_crit/DIAS,2), 'R$/dia', '13m', 'Vendas', 'receita histórica desses SKUs / dias')
reg('ruptura_curvaA_receita_ano', round(rec_crit*ANU,2), 'R$/ano', 'anualizado', 'Vendas', 'receita_13m * 365/dias (RISCO, não perda incorrida)')

sku_qtd = v.groupby('sku_id')['quantidade'].sum()
est2 = estoque.merge(sku_qtd.rename('q'), on='sku_id', how='left').fillna({'q':0})
est2['cob'] = est2['estoque_disponivel'] / (est2['q']/DIAS).replace(0,0.001)
reg('pct_skus_2anos_cobertura', round(100*(est2['cob']>730).mean(),2), '%', '13m', 'Estoque × Vendas', 'share de SKUs com cobertura > 730 dias')
reg('capital_imobilizado_rs', round((estoque['estoque_disponivel']*estoque['custo_unitario']).sum(),2), 'R$', 'snapshot', 'Estoque', 'estoque_disponivel * custo_unitario — DESCARTADO (artefato de foto de estoque)')

seg_risco = ['Churn','Em Risco','Hibernando']
reg('rfm_ltv_exposto', round(clientes.loc[clientes['segmento_rfm'].isin(seg_risco),'ltv_acumulado'].sum(),2), 'R$', 'cadastro', 'Clientes.ltv_acumulado (segmentos em risco)', 'soma LTV — EXPOSIÇÃO, não perda')
reg('rfm_clientes_exposto', int(clientes['segmento_rfm'].isin(seg_risco).sum()), 'clientes', 'cadastro', 'Clientes', 'contagem de segmentos em risco')
None

  ruptura_curvaA_qtd                     = 417 SKUs
  ruptura_curvaA_receita_dia             = 5748.47 R$/dia
  ruptura_curvaA_receita_ano             = 2098192.64 R$/ano
  pct_skus_2anos_cobertura               = 91.7 %
  capital_imobilizado_rs                 = 296549794.93 R$
  rfm_ltv_exposto                        = 27801944.73 R$
  rfm_clientes_exposto                   = 6993 clientes


## 7. Tamanhos de efeito canônicos

Reaproveita a lógica do notebook 03 — o que decide a leitura não é o p-valor (com n grande
quase tudo é significativo), é o tamanho de efeito.

In [8]:
def eps2(H,n,k): return (H-k+1)/(n-k)
d = v.dropna(subset=['desconto_pct','margem_pct'])
reg('es_desconto_r2', round(stats.pearsonr(d['desconto_pct'], d['margem_pct'])[0]**2,4), 'r²', '13m', 'Vendas', 'pearson(desconto,margem)² — maior driver de margem')
canais = v['canal'].dropna().unique().tolist()
Hc,_ = stats.kruskal(*[v.loc[v['canal']==c,'margem_pct'].dropna() for c in canais])
reg('es_canal_eps2', round(eps2(Hc,len(v),len(canais)),4), 'eps2', '13m', 'Vendas', 'kruskal margem~canal')
segs = clientes['segmento_rfm'].dropna().unique().tolist()
Hs,_ = stats.kruskal(*[clientes.loc[clientes['segmento_rfm']==s,'ltv_acumulado'].dropna() for s in segs])
reg('es_rfm_eps2', round(eps2(Hs,len(clientes),len(segs)),4), 'eps2', 'cadastro', 'Clientes', 'kruskal ltv~segmento_rfm')
None

  es_desconto_r2                         = 0.2859 r²
  es_canal_eps2                          = 0.0315 eps2
  es_rfm_eps2                            = 0.3809 eps2


## 10. Identidades contábeis da base

Antes de qualquer leitura financeira, é preciso saber se as colunas de Vendas fecham entre
si. Duas identidades sustentam tudo o que vem depois:

- `receita_liquida = receita_bruta − desconto_reais`
- `margem_contribuicao = receita_liquida − custo_produto − custo_frete`

Se as duas fecham, **"R$ 1 de desconto é R$ 1 de margem cedida" deixa de ser premissa e
passa a ser aritmética da própria base** — o que muda o peso do argumento de desconto de
"suposição generosa" para "identidade verificável".

In [9]:
d_rl = (v['receita_bruta'] - v['desconto_reais'] - v['receita_liquida']).abs()
d_mc = (v['receita_liquida'] - v['custo_produto'] - v['custo_frete'] - v['margem_contribuicao']).abs()
reg('identidade_receita_desvio_max', round(float(d_rl.max()), 4), 'R$', '13m', 'Vendas',
    'max|receita_bruta - desconto_reais - receita_liquida|')
reg('identidade_margem_desvio_max', round(float(d_mc.max()), 4), 'R$', '13m', 'Vendas',
    'max|receita_liquida - custo_produto - custo_frete - margem_contribuicao|')
reg('identidade_linhas_fora_1centavo', int(((d_rl > 0.01) | (d_mc > 0.01)).sum()), 'linhas', '13m', 'Vendas',
    'no de linhas que violam qualquer das duas identidades')
None

  identidade_receita_desvio_max          = 0.0 R$
  identidade_margem_desvio_max           = 0.0 R$
  identidade_linhas_fora_1centavo        = 0 linhas


## 11. Integridade das bases — o que cada base permite e o que ela bloqueia

Estas métricas não são "achados de negócio": são o **limite de factibilidade** de qualquer
solução construída sobre cada base. É o que separa uma frente executável de uma frente que
depende de consertar instrumentação antes.

In [10]:
# --- Marketing: a base declara volumes incompatíveis com Vendas (trava H4/H5) ---
reg('mkt_campanhas', int(len(marketing)), 'campanhas', 'cadastro', 'Marketing.csv', 'contagem de linhas')
reg('mkt_investimento_declarado_rs', round(marketing['investimento_reais'].sum(), 2), 'R$', 'cadastro',
    'Marketing.investimento_reais', 'soma - incompatível com a escala de Vendas')
reg('mkt_receita_declarada_vs_real_x', round(marketing['receita_gerada'].sum() / rb, 1), 'x', 'cadastro',
    'Marketing x Vendas', 'receita_gerada declarada / receita_bruta real')
reg('mkt_conversoes_vs_pedidos_x', round(marketing['conversoes'].sum() / len(v), 0), 'x', 'cadastro',
    'Marketing x Vendas', 'conversoes declaradas / pedidos reais')
reg('mkt_modelos_atribuicao_qtd', int(marketing['atribuicao'].nunique()), 'modelos', 'cadastro',
    'Marketing.atribuicao', 'modelos de atribuicao coexistindo na mesma base')

# --- Estoque: é uma foto, sem histórico de dias sem estoque ---
sku_qtd_11 = v.groupby('sku_id')['quantidade'].sum()
est11 = estoque.merge(sku_qtd_11.rename('q'), on='sku_id', how='left').fillna({'q': 0})
est11['cob'] = est11['estoque_disponivel'] / (est11['q'] / DIAS).replace(0, 0.001)
cmv_ano = (v['custo_produto'].sum() + v['custo_frete'].sum()) * ANU
reg('estoque_cobertura_mediana_dias', round(float(est11['cob'].median()), 0), 'dias', '13m', 'Estoque x Vendas',
    'mediana de estoque_disponivel / consumo diário')
reg('estoque_unidades_vs_vendas_x', round(estoque['estoque_disponivel'].sum() / v['quantidade'].sum(), 1), 'x', '13m',
    'Estoque x Vendas', 'unidades em estoque / unidades vendidas na janela')
reg('estoque_cobertura_anos_cmv', round((estoque['estoque_disponivel'] * estoque['custo_unitario']).sum() / cmv_ano, 1),
    'anos', 'snapshot', 'Estoque x Vendas', 'capital imobilizado / CMV anual (custo_produto + custo_frete)')

# --- Atendimento: volume total, base de 36 meses ---
reg('atend_tickets_total', int(len(atendimento)), 'tickets', '36m', 'Atendimento.csv', 'contagem')
reg('atend_tickets_ano', round(len(atendimento) / 3, 1), 'tickets/ano', '36m/3', 'Atendimento.csv', 'total / 3')
None

  mkt_campanhas                          = 3500 campanhas
  mkt_investimento_declarado_rs          = 210414341.09 R$
  mkt_receita_declarada_vs_real_x        = 42.8 x
  mkt_conversoes_vs_pedidos_x            = 4153.0 x
  mkt_modelos_atribuicao_qtd             = 3 modelos
  estoque_cobertura_mediana_dias         = 5339.0 dias
  estoque_unidades_vs_vendas_x           = 14.6 x
  estoque_cobertura_anos_cmv             = 36.8 anos
  atend_tickets_total                    = 35840 tickets
  atend_tickets_ano                      = 11946.7 tickets/ano


## 12. Ruptura — três leituras, e por que a comparável é a terceira

A ruptura é a única frente do diagnóstico medida em **receita**. Todas as outras são margem
ou custo. Somar ou ranquear receita contra margem superdimensiona esta frente.

A correção **não** é aplicar uma taxa média de margem sobre a receita — isso seria um proxy.
A correção é somar a **margem observada dos mesmos SKUs**, que está na base.

Segunda correção, igualmente material: `status_disponibilidade` distingue **`Ruptura`** de
**`Estoque Crítico`**. Um SKU em estoque crítico ainda tem estoque — é exposição, não perda
corrente. Tratar os dois como a mesma coisa infla a frente em quase 7x.

In [11]:
skus_crit  = set(crit_a['sku_id'])
sku_mc     = v.groupby('sku_id')['margem_contribuicao'].sum()
sku_mc_rea = realizado.groupby('sku_id')['margem_contribuicao'].sum()

mc_crit  = sku_mc.loc[sku_mc.index.isin(skus_crit)].sum()
mcr_crit = sku_mc_rea.loc[sku_mc_rea.index.isin(skus_crit)].sum()
reg('ruptura_curvaA_margem_contrib_ano', round(mc_crit * ANU, 2), 'R$/ano', 'anualizado', 'Vendas x Estoque',
    'margem de contribuicao propria dos SKUs criticos * 365/dias (sem taxa media)')
reg('ruptura_curvaA_margem_realizada_ano', round(mcr_crit * ANU, 2), 'R$/ano', 'anualizado', 'Vendas x Estoque',
    'margem REALIZADA propria desses SKUs * 365/dias - LEITURA COMPARAVEL as demais frentes')

so_rup = set(estoque.loc[estoque['status_disponibilidade'] == 'Ruptura', 'sku_id']) & set(skus_a)
so_cri = set(estoque.loc[estoque['status_disponibilidade'] == 'Estoque Crítico', 'sku_id']) & set(skus_a)
reg('ruptura_em_ruptura_skus', int(len(so_rup)), 'SKUs', 'snapshot', 'Estoque', "curva A com status == 'Ruptura'")
reg('ruptura_estoque_critico_skus', int(len(so_cri)), 'SKUs', 'snapshot', 'Estoque', "curva A com status == 'Estoque Critico'")
reg('ruptura_em_ruptura_margem_realizada_ano',
    round(sku_mc_rea.loc[sku_mc_rea.index.isin(so_rup)].sum() * ANU, 2), 'R$/ano', 'anualizado', 'Vendas x Estoque',
    'margem realizada so dos SKUs efetivamente em ruptura - PERDA CORRENTE')
None

  ruptura_curvaA_margem_contrib_ano      = 1150547.5 R$/ano
  ruptura_curvaA_margem_realizada_ano    = 867128.17 R$/ano
  ruptura_em_ruptura_skus                = 54 SKUs
  ruptura_estoque_critico_skus           = 363 SKUs
  ruptura_em_ruptura_margem_realizada_ano = 120022.3 R$/ano


## 13. Decomposição da margem que não virou caixa (H9)

O maior número do diagnóstico (R$ 2,40 mi/ano) é rotulado como "devolução", mas mede
**tudo que não virou caixa**: a definição de margem realizada é `aprovado E não devolvido`.
Cancelamento e pagamento pendente sempre estiveram dentro dele.

Isto **não abre hipótese nova** — decompõe a evidência de uma hipótese já Confirmada. Sem a
decomposição, a frente não tem endereço de intervenção: "devolução" e "pagamento pendente"
se resolvem por caminhos completamente diferentes.

In [12]:
dev   = v['devolvido']
nao_dev = ~dev

reg('h9_devolucao_margem_ano', round(v.loc[dev, 'margem_contribuicao'].sum() * ANU, 2), 'R$/ano', 'anualizado',
    'Vendas (devolvido)', 'margem dos pedidos devolvidos * 365/dias')
reg('h9_cancelado_margem_ano',
    round(v.loc[nao_dev & (v['status_pagamento'] == 'Cancelado'), 'margem_contribuicao'].sum() * ANU, 2),
    'R$/ano', 'anualizado', 'Vendas (cancelado, nao devolvido)', 'margem * 365/dias')
reg('h9_aguardando_margem_ano',
    round(v.loc[nao_dev & (v['status_pagamento'] == 'Aguardando'), 'margem_contribuicao'].sum() * ANU, 2),
    'R$/ano', 'anualizado', 'Vendas (aguardando, nao devolvido)', 'margem * 365/dias')

MOT = {'Produto com defeito': 'defeito', 'Tamanho errado': 'tamanho', 'Atraso na entrega': 'atraso',
       'Não gostei': 'naogostei', 'Arrependimento': 'arrependimento'}
por_motivo = v.loc[dev].groupby('motivo_devolucao')['margem_contribuicao'].sum() * ANU
for rotulo, chave in MOT.items():
    if rotulo in por_motivo.index:
        reg('h9_dev_' + chave + '_ano', round(float(por_motivo[rotulo]), 2), 'R$/ano', 'anualizado',
            'Vendas.motivo_devolucao', "margem devolvida por '" + rotulo + "' * 365/dias")

ender = ['Produto com defeito', 'Tamanho errado', 'Atraso na entrega']
reg('h9_dev_enderecavel_ano', round(float(por_motivo[por_motivo.index.isin(ender)].sum()), 2), 'R$/ano', 'anualizado',
    'Vendas.motivo_devolucao', 'defeito + tamanho + atraso (motivos com causa operacional identificavel)')
reg('h9_dev_nao_enderecavel_ano', round(float(por_motivo[~por_motivo.index.isin(ender)].sum()), 2), 'R$/ano',
    'anualizado', 'Vendas.motivo_devolucao', 'nao gostei + arrependimento (preferencia do cliente)')

tx_canal = v.groupby('canal')['devolvido'].mean() * 100
tx_categ = v.groupby('categoria')['devolvido'].mean() * 100
reg('devolucao_tx_global_pct', round(100 * dev.mean(), 2), '%', '13m', 'Vendas.devolvido', 'share global de devolucao')
reg('devolucao_tx_amplitude_canal_pp', round(float(tx_canal.max() - tx_canal.min()), 2), 'p.p.', '13m', 'Vendas',
    'max-min da taxa de devolucao entre canais (~0 -> devolucao e sistemica, nao concentrada)')
reg('devolucao_tx_amplitude_categoria_pp', round(float(tx_categ.max() - tx_categ.min()), 2), 'p.p.', '13m', 'Vendas',
    'max-min da taxa de devolucao entre categorias')
None

  h9_devolucao_margem_ano                = 1425548.37 R$/ano
  h9_cancelado_margem_ano                = 653067.07 R$/ano
  h9_aguardando_margem_ano               = 320759.97 R$/ano
  h9_dev_defeito_ano                     = 375448.62 R$/ano
  h9_dev_tamanho_ano                     = 342043.67 R$/ano
  h9_dev_atraso_ano                      = 272171.86 R$/ano
  h9_dev_naogostei_ano                   = 236205.07 R$/ano
  h9_dev_arrependimento_ano              = 199679.15 R$/ano
  h9_dev_enderecavel_ano                 = 989664.15 R$/ano
  h9_dev_nao_enderecavel_ano             = 435884.22 R$/ano
  devolucao_tx_global_pct                = 14.87 %
  devolucao_tx_amplitude_canal_pp        = 0.9 p.p.
  devolucao_tx_amplitude_categoria_pp    = 1.18 p.p.


## 14. Desconto — sobreposição medida e recuperável por política

Duas correções sobre o R$ 1,53 mi:

**(a) Sobreposição, agora medida.** Parte do desconto foi concedida em pedidos que nunca
viraram caixa — esse pedaço já está contado na H9. Até aqui a sobreposição era declarada
("não somar"); passa a ser um número, o que permite consolidar em vez de só ressalvar.

**(b) Recuperável ≠ total.** O R$ 1,53 mi é o desconto inteiro — equivale a supor que a
política seria *zerar* desconto, o que nenhuma diretoria faz. O número acionável é o
**excedente acima de um teto**. Como o nb04 mostra que unidades por pedido não respondem a
desconto (amplitude de 0,14 unidade entre faixas), o excedente é margem cedida sem
contrapartida de volume — recuperável 1:1, pela identidade verificada na seção 10.

In [13]:
d_nao = v.loc[~v.index.isin(realizado.index), 'desconto_reais'].sum()
d_rea = realizado['desconto_reais'].sum()
reg('desconto_sobreposto_nao_realizado_ano', round(d_nao * ANU, 2), 'R$/ano', 'anualizado', 'Vendas',
    'desconto concedido em pedidos que nao viraram caixa - DUPLA CONTAGEM com margem_gap')
reg('desconto_sobreposto_pct', round(100 * d_nao / v['desconto_reais'].sum(), 1), '%', '13m', 'Vendas',
    'share do desconto que esta dentro da H9')
reg('desconto_adicional_realizado_ano', round(d_rea * ANU, 2), 'R$/ano', 'anualizado', 'Vendas',
    'desconto em pedidos que viraram caixa - incremento REAL sobre a H9, sem dupla contagem')

for teto in (0.20, 0.15):
    acima = realizado[realizado['desconto_pct'] > teto]
    exced = ((acima['desconto_pct'] - teto) * acima['receita_bruta']).sum()
    tag = str(int(teto * 100))
    reg('desconto_excedente_teto' + tag + '_ano', round(exced * ANU, 2), 'R$/ano', 'anualizado', 'Vendas',
        'soma de (desconto_pct - ' + tag + '%) * receita_bruta nos pedidos acima do teto * 365/dias')
    reg('desconto_pedidos_acima_' + tag + 'pct', int(len(acima)), 'pedidos', '13m', 'Vendas',
        'pedidos realizados com desconto acima de ' + tag + '%')
None

  desconto_sobreposto_nao_realizado_ano  = 379560.48 R$/ano
  desconto_sobreposto_pct                = 24.8 %
  desconto_adicional_realizado_ano       = 1152316.29 R$/ano
  desconto_excedente_teto20_ano          = 291186.59 R$/ano
  desconto_pedidos_acima_20pct           = 4320 pedidos
  desconto_excedente_teto15_ano          = 457047.65 R$/ano
  desconto_pedidos_acima_15pct           = 5342 pedidos


## 15. Cobertura e granularidade — o que cada base consegue responder

Duas bases passam em integridade agregada mas **falham na granularidade** que uma solução
exigiria. Estas primitivas existem para que a factibilidade de qualquer módulo seja
calculada a partir do dado, e não estimada no olho:

- **Clientes** casa 100% com Vendas, mas Vendas só contém uma fração dos clientes do
  cadastro — qualquer métrica por cliente calculada a partir de Vendas é irrepresentativa.
- **Atendimento** tem volume alto, mas `texto_cliente` é um conjunto pequeno de frases
  repetidas — o que limita o que um classificador de linguagem pode aprender.

In [14]:
ids_vendas = v['customer_id'].nunique()
reg('clientes_ids_em_vendas', int(ids_vendas), 'clientes', '13m', 'Vendas.customer_id', 'nunique')
reg('clientes_base_total', int(len(clientes)), 'clientes', 'cadastro', 'Clientes.csv', 'contagem')
reg('clientes_cobertura_pct', round(100 * ids_vendas / len(clientes), 2), '%', '13m', 'Vendas x Clientes',
    'clientes presentes em Vendas / base de clientes - granularidade por cliente')
reg('clientes_pedidos_por_id', round(len(v) / ids_vendas, 1), 'pedidos/cliente', '13m', 'Vendas',
    'pedidos / customer_id distintos - implausivel, sinaliza id sintetico')
reg('clientes_match_vendas_pct', round(100 * v['customer_id'].isin(clientes['customer_id']).mean(), 2), '%', '13m',
    'Vendas x Clientes', 'share de pedidos cujo customer_id existe no cadastro')

reg('atend_frases_distintas', int(atendimento['texto_cliente'].nunique()), 'frases', '36m',
    'Atendimento.texto_cliente', 'nunique - teto do que um classificador de linguagem pode aprender')
reg('atend_frases_por_ticket', round(atendimento['texto_cliente'].nunique() / len(atendimento), 5), 'frases/ticket',
    '36m', 'Atendimento', 'frases distintas / tickets - proxima de 0 -> texto e template, nao linguagem livre')
None

  clientes_ids_em_vendas                 = 346 clientes
  clientes_base_total                    = 15000 clientes
  clientes_cobertura_pct                 = 2.31 %
  clientes_pedidos_por_id                = 80.2 pedidos/cliente
  clientes_match_vendas_pct              = 100.0 %
  atend_frases_distintas                 = 30 frases
  atend_frases_por_ticket                = 0.00084 frases/ticket


## 8. Séries para os gráficos da dashboard

Arrays prontos para a HTML — mantidos na mesma fonte única, para que nenhum gráfico da
dashboard calcule número por conta própria.

In [15]:
SERIES = {}
# faixas de desconto (margem média por pedido, unidades, nº pedidos)
SERIES['desconto_faixas'] = [
    {'faixa': str(ix), 'margem_pct': round(100*r['margem_pct'],1),
     'unidades': round(r['unidades'],2), 'pedidos': int(r['pedidos'])}
    for ix, r in por_faixa.iterrows()]

# margem agregada e % de pedidos que pagam frete, por canal
canal_stats = v.groupby('canal').apply(lambda g: pd.Series({
    'margem_agg': round(100*g['margem_contribuicao'].sum()/g['receita_liquida'].sum(),1),
    'frete_pedidos_pct': round(100*(g['custo_frete']>0).mean(),0)}), include_groups=False)
SERIES['canal_margem'] = [{'canal': c, **canal_stats.loc[c].to_dict()} for c in canal_stats.index]

# ponte de margem (waterfall): contábil -> tira cancelados/aguardando -> tira devolvidos -> realizada
mc_total = v['margem_contribuicao'].sum()
m_nao_aprov = v.loc[v['status_pagamento']!='Aprovado','margem_contribuicao'].sum()
m_devol_aprov = v.loc[(v['devolvido']) & (v['status_pagamento']=='Aprovado'),'margem_contribuicao'].sum()
SERIES['margem_bridge'] = {
    'contabil_rs': round(mc_total,0), 'cancel_aguard_rs': round(m_nao_aprov,0),
    'devolucao_rs': round(m_devol_aprov,0), 'realizada_rs': round(mc_total-m_nao_aprov-m_devol_aprov,0),
    'contabil_pct': round(100*mc_total/rl,1), 'realizada_pct': round(100*(mc_total-m_nao_aprov-m_devol_aprov)/rl,1)}

# natureza dos chamados (36m)
tot_tk = len(atendimento)
falha_cats = ['Onde está meu pedido?','Defeito','Pagamento não aprovado']
duvida_troca = ['Dúvida Técnica','Troca de Tamanho']
SERIES['chamados_natureza'] = {
    'falha': round(100*atendimento['categoria_problema'].isin(falha_cats).mean(),1),
    'duvida_troca': round(100*atendimento['categoria_problema'].isin(duvida_troca).mean(),1),
    'elogio': round(100*(atendimento['categoria_problema']=='Elogio').mean(),1)}

# custo e csat por canal de atendimento
atend_canal = atendimento.groupby('canal_entrada').agg(
    custo=('custo_operacional_ticket','mean'), csat=('nota_csat','mean'), n=('ticket_id','size'))
SERIES['atend_canais'] = [{'canal': c, 'custo': round(atend_canal.loc[c,'custo'],2),
    'csat': round(atend_canal.loc[c,'csat'],2), 'share': round(100*atend_canal.loc[c,'n']/tot_tk,1)}
    for c in atend_canal.sort_values('custo').index]

# LTV mediano por segmento RFM
ltv_seg = clientes.groupby('segmento_rfm')['ltv_acumulado'].median().sort_values(ascending=False)
SERIES['ltv_rfm'] = [{'segmento': s, 'ltv': round(ltv_seg[s],0)} for s in ltv_seg.index]

for k, s in SERIES.items():
    print(k, '->', (len(s) if isinstance(s,list) else s))
None

desconto_faixas -> 6
canal_margem -> 7
margem_bridge -> {'contabil_rs': np.float64(10270437.0), 'cancel_aguard_rs': np.float64(1212009.0), 'devolucao_rs': np.float64(1351707.0), 'realizada_rs': np.float64(7706720.0), 'contabil_pct': np.float64(54.4), 'realizada_pct': np.float64(40.8)}
chamados_natureza -> {'falha': np.float64(60.1), 'duvida_troca': np.float64(29.8), 'elogio': np.float64(10.1)}
atend_canais -> 5
ltv_rfm -> 6


In [16]:
# --- séries acrescentadas pelas seções 12-14 (decomposição, ruptura e tetos de desconto) ---
SERIES['h9_decomposicao'] = [
    {'buraco': 'Devolução',            'rs_ano': round(CANON['h9_devolucao_margem_ano']['valor'], 0)},
    {'buraco': 'Pagamento cancelado',  'rs_ano': round(CANON['h9_cancelado_margem_ano']['valor'], 0)},
    {'buraco': 'Pagamento aguardando', 'rs_ano': round(CANON['h9_aguardando_margem_ano']['valor'], 0)}]

SERIES['devolucao_motivos'] = [
    {'motivo': m, 'rs_ano': round(float(por_motivo[m]), 0), 'enderecavel': bool(m in ender)}
    for m in por_motivo.sort_values(ascending=False).index]

SERIES['ruptura_leituras'] = [
    {'leitura': 'Receita histórica',      'rs_ano': round(CANON['ruptura_curvaA_receita_ano']['valor'], 0)},
    {'leitura': 'Margem de contribuição', 'rs_ano': round(CANON['ruptura_curvaA_margem_contrib_ano']['valor'], 0)},
    {'leitura': 'Margem realizada',       'rs_ano': round(CANON['ruptura_curvaA_margem_realizada_ano']['valor'], 0)},
    {'leitura': 'Só SKUs já em ruptura',  'rs_ano': round(CANON['ruptura_em_ruptura_margem_realizada_ano']['valor'], 0)}]

SERIES['desconto_tetos'] = [
    {'cenario': 'Desconto total (teto 1:1)', 'rs_ano': round(CANON['desconto_total_rs_ano']['valor'], 0)},
    {'cenario': 'Excedente acima de 15%',    'rs_ano': round(CANON['desconto_excedente_teto15_ano']['valor'], 0)},
    {'cenario': 'Excedente acima de 20%',    'rs_ano': round(CANON['desconto_excedente_teto20_ano']['valor'], 0)}]

for k in ['h9_decomposicao', 'devolucao_motivos', 'ruptura_leituras', 'desconto_tetos']:
    print(k, '->', SERIES[k])
None

h9_decomposicao -> [{'buraco': 'Devolução', 'rs_ano': np.float64(1425548.0)}, {'buraco': 'Pagamento cancelado', 'rs_ano': np.float64(653067.0)}, {'buraco': 'Pagamento aguardando', 'rs_ano': np.float64(320760.0)}]
devolucao_motivos -> [{'motivo': 'Produto com defeito', 'rs_ano': 375449.0, 'enderecavel': True}, {'motivo': 'Tamanho errado', 'rs_ano': 342044.0, 'enderecavel': True}, {'motivo': 'Atraso na entrega', 'rs_ano': 272172.0, 'enderecavel': True}, {'motivo': 'Não gostei', 'rs_ano': 236205.0, 'enderecavel': False}, {'motivo': 'Arrependimento', 'rs_ano': 199679.0, 'enderecavel': False}]
ruptura_leituras -> [{'leitura': 'Receita histórica', 'rs_ano': np.float64(2098193.0)}, {'leitura': 'Margem de contribuição', 'rs_ano': np.float64(1150548.0)}, {'leitura': 'Margem realizada', 'rs_ano': np.float64(867128.0)}, {'leitura': 'Só SKUs já em ruptura', 'rs_ano': np.float64(120022.0)}]
desconto_tetos -> [{'cenario': 'Desconto total (teto 1:1)', 'rs_ano': np.float64(1531877.0)}, {'cenario': 'Ex

## 8.1 Duas métricas que faltavam — resolvendo os dois pontos em aberto da v3

Dois itens ficaram como "ainda em aberto" na primeira leitura da v3: (a) o quanto a composição de receita por categoria realmente varia mês a mês, e (b) qual fração dos chamados de atendimento as frases mais frequentes já cobrem. Nenhuma das duas tinha célula própria na sequência `01` a `08` — seção acrescentada na v4 para fechar essa lacuna, na mesma base e com a mesma metodologia do resto do notebook.

In [33]:
# --- Composição de receita por categoria: amplitude mensal, não só a média do período ---
# A média do período (seção 2, "por_categoria") mostra uma composição estável (spread de
# ~0,3 p.p.). Mas a média mensal esconde variação dentro do período: a pergunta certa é
# "o quanto a participação de cada categoria oscila mês a mês", não "qual é a média".
rl_mes_cat = v.groupby([v['data_pedido'].dt.to_period('M').astype(str), 'categoria'])['receita_liquida'].sum().unstack('categoria')
participacao_mensal = 100 * rl_mes_cat.div(rl_mes_cat.sum(axis=1), axis=0)
amplitude_por_categoria = (participacao_mensal.max() - participacao_mensal.min()).sort_values(ascending=False)
print('Amplitude mensal de participação na receita líquida, por categoria (p.p.):')
print(amplitude_por_categoria.round(2).to_string())
reg('receita_categoria_amplitude_mensal_pp', round(float(amplitude_por_categoria.max()), 2), 'p.p.', '13m',
    'Vendas.categoria x mês', 'maior categoria: max_mês(participação %) - min_mês(participação %)')

print()

# --- Atendimento: cobertura das frases mais frequentes ---
freq_frases = atendimento['texto_cliente'].value_counts()
top10_pct = 100 * freq_frases.head(10).sum() / len(atendimento)
print('As 10 frases mais frequentes cobrem {:.1f}% dos {:,} tickets (de {} frases distintas).'.format(
      top10_pct, len(atendimento), atendimento['texto_cliente'].nunique()))
reg('atend_top10_frases_pct', round(top10_pct, 1), '%', '36m', 'Atendimento.texto_cliente',
    'share de tickets cobertos pelas 10 frases mais frequentes')

Amplitude mensal de participação na receita líquida, por categoria (p.p.):
categoria
Beleza        4.39
Lifestyle     4.37
Moda          4.10
Acessórios    3.92
  receita_categoria_amplitude_mensal_pp  = 4.39 p.p.

As 10 frases mais frequentes cobrem 48.2% dos 35,840 tickets (de 30 frases distintas).
  atend_top10_frases_pct                 = 48.2 %

## 9. Exportação

Grava `outputs/numeros_canonicos.json` (a fonte que o `05`, o `07` e a HTML consomem) e
uma tabela legível `outputs/numeros_canonicos.md`.

In [17]:
meta = {'gerado_por': '06_numeros_canonicos.ipynb', 'janela_vendas_dias': int(DIAS),
        'fator_anualizacao_vendas': round(ANU,5), 'janela_atendimento_meses': 36,
        'projecao_base_completa': False, 'n_metricas': len(CANON)}
payload = {'_meta': meta, 'metricas': CANON, 'series': SERIES}
Path('outputs/numeros_canonicos.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

linhas = ['# Números canônicos — Projeto Vértice', '',
          f"Janela de vendas: {DIAS} dias · fator de anualização {ANU:.5f} · atendimento 36 meses · sem projeção ×2,88.", '',
          '| Métrica | Valor | Unidade | Janela | Fonte |', '|---|---|---|---|---|']
for k,c in CANON.items():
    linhas.append(f"| `{k}` | {c['valor']} | {c['unidade']} | {c['janela']} | {c['fonte']} |")
Path('outputs/numeros_canonicos.md').write_text('\n'.join(linhas), encoding='utf-8')
print(f'{len(CANON)} métricas exportadas para outputs/numeros_canonicos.json e .md')

98 métricas exportadas para outputs/numeros_canonicos.json e .md
